# Sentinel · synthetic questions for the reranker fine-tune

Runs `scripts/generate_queries.py` on a free GPU: [doc2query/msmarco-t5-base-v1](https://huggingface.co/doc2query/msmarco-t5-base-v1) (pinned revision) writes 4 questions for each of the ~4,900 RFC passage windows in `data/reranker/windows.jsonl.gz`.

1. **Runtime → Change runtime type → T4 GPU**
2. **Runtime → Run all** (about 10 minutes)
3. The last cell downloads `queries.jsonl.gz` and its `.meta.json` — put both in `data/reranker/` in the repo.

If the runtime disconnects, run all again: finished batches are skipped, and resumed output is identical to an unbroken run.

In [ ]:
!nvidia-smi -L || echo 'No GPU: Runtime > Change runtime type > T4 GPU'

In [ ]:
!test -d Sentinel || git clone --depth 1 --branch feat/reranker-finetune https://github.com/krishivsaini/Sentinel.git
%cd Sentinel
!git log --oneline -1

In [ ]:
import torch, transformers
print('torch', torch.__version__, '| transformers', transformers.__version__, '| cuda', torch.cuda.is_available())

In [ ]:
!python scripts/generate_queries.py \
    --windows data/reranker/windows.jsonl.gz \
    --out queries.jsonl.gz \
    --batch-size 64 --device cuda

In [ ]:
import gzip, json, itertools
rows = [json.loads(l) for l in gzip.open('queries.jsonl.gz', 'rt')]
print(len(rows), 'windows,', sum(len(r['queries']) for r in rows), 'questions')
for r in itertools.islice(rows, 0, 2000, 400):
    print(r['window_id'], r['queries'])

In [ ]:
from google.colab import files
files.download('queries.jsonl.gz')
files.download('queries.jsonl.gz.meta.json')